# 🤖 SVM Classification: Kernel Comparison
## Linear vs Polynomial vs RBF

**College ML Course Project** | Python 3.x | scikit-learn, matplotlib, seaborn

> **How to run:** Go to **Runtime → Run all** (or press `Ctrl+F9`). All outputs and plots will appear inline below each cell. All PNGs are saved to the `outputs/` folder in Colab's filesystem.

---

## 📖 Background: What is a Kernel in SVM?

Support Vector Machine (SVM) tries to find the **best hyperplane** that separates two classes with the maximum margin.  
When data is **not linearly separable** in the original feature space, SVM uses a **kernel function** to implicitly map data into a higher-dimensional space where a linear separator *does* exist — this is the **Kernel Trick**.

| Kernel | Formula | Key Property |
|---|---|---|
| **Linear** | $K(x,z) = x \cdot z$ | Fastest; works only on linearly separable data |
| **Polynomial** | $K(x,z) = (\gamma x \cdot z + r)^d$ | Models polynomial feature interactions |
| **RBF (Gaussian)** | $K(x,z) = \exp(-\gamma \|x-z\|^2)$ | Most flexible; works on arbitrarily shaped boundaries |

---

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 1 — Setup & Imports
# All packages are pre-installed on Colab. No pip installs needed.
# ════════════════════════════════════════════════════════════════════

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets       import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing  import StandardScaler
from sklearn.svm            import SVC
from sklearn.metrics        import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Create output directory for saving plots
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Consistent plot style throughout the notebook
plt.rcParams.update({
    'figure.dpi'     : 120,
    'font.family'    : 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_theme(style='whitegrid', font_scale=1.0)

# Colour scheme
CMAP_LIGHT   = 'RdYlBu'
CLASS_COLORS = ['#EF5350', '#42A5F5']   # red=class 0, blue=class 1

print('✅ Setup complete. All libraries loaded.')
print(f'   Outputs will be saved to: ./{OUTPUT_DIR}/')

---
# Part 1 — Data Preparation

## 1A. Synthetic Dataset: `make_moons`

### Why does Linear SVM fail on non-linear data?

> A linear kernel computes $K(x, z) = x \cdot z$ — it only finds a **straight hyperplane** in the *original* feature space. The `make_moons` dataset creates two interleaving half-circles that *cannot* be separated by any straight line, no matter how you adjust the margin parameter C. This fundamentally motivates the need for non-linear kernels.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 2 — Load & Preprocess Synthetic Data (make_moons)
# ════════════════════════════════════════════════════════════════════

def load_synthetic_data(n_samples=500, noise=0.25, random_state=42):
    """
    Generate a non-linearly separable dataset using make_moons.
    Returns scaled train/test splits and the fitted scaler.

    Preprocessing:
      - 80/20 stratified train/test split
      - StandardScaler fit on train, applied to both
        (avoids data leakage from the test set into scaling)
    """
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=random_state)

    # Stratified split: ensures equal class ratio in both sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )

    # Standardize: SVM is distance-based, so scale matters
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)   # fit+transform on train
    X_test_sc  = scaler.transform(X_test)         # transform only on test

    return X_train_sc, X_test_sc, y_train, y_test, scaler


# Load the data
X_tr_syn, X_te_syn, y_tr_syn, y_te_syn, scaler_syn = load_synthetic_data()

print(f'Synthetic dataset loaded:')
print(f'  Train : {X_tr_syn.shape} | Class distribution: {np.bincount(y_tr_syn)}')
print(f'  Test  : {X_te_syn.shape} | Class distribution: {np.bincount(y_te_syn)}')

# Quick visualisation of raw data
fig, ax = plt.subplots(figsize=(6, 4))
for cls, color, label in zip([0, 1], CLASS_COLORS, ['Class 0', 'Class 1']):
    mask = y_tr_syn == cls
    ax.scatter(X_tr_syn[mask, 0], X_tr_syn[mask, 1],
               c=color, edgecolors='white', linewidths=0.5,
               s=35, label=label, alpha=0.85)
ax.set_title('make_moons — Training Data (scaled)', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dataset_moons_raw.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: dataset_moons_raw.png')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 3 — Load & Preprocess Real Data (Breast Cancer Wisconsin)
# ════════════════════════════════════════════════════════════════════

def load_real_data(random_state=42):
    """
    Load the Breast Cancer Wisconsin dataset.
    Returns:
      - Full 30-feature version (for accurate metric evaluation)
      - 2-D slice (features 0 & 1) for decision boundary visualisation
    """
    data = load_breast_cancer()
    X, y = data.data, data.target

    # Full feature set
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # 2-D slice for plots
    X_2d = X[:, :2]
    X_tr2d, X_te2d, _, _ = train_test_split(
        X_2d, y, test_size=0.2, random_state=random_state, stratify=y
    )
    sc2d = StandardScaler()
    X_tr2d = sc2d.fit_transform(X_tr2d)
    X_te2d = sc2d.transform(X_te2d)

    return (X_train_sc, X_test_sc, y_train, y_test,
            X_tr2d, X_te2d, data.feature_names)


(X_tr_bc, X_te_bc, y_tr_bc, y_te_bc,
 X_tr_bc2d, X_te_bc2d, bc_features) = load_real_data()

print('Breast Cancer Wisconsin dataset loaded:')
print(f'  Full features: {X_tr_bc.shape} train, {X_te_bc.shape} test')
print(f'  2-D features : {X_tr_bc2d.shape} train, {X_te_bc2d.shape} test')
print(f'  Classes: 0=Malignant, 1=Benign')
print(f'  Class dist — Train: {np.bincount(y_tr_bc)}, Test: {np.bincount(y_te_bc)}')

---
# Part 2 — Training the Three SVM Models

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 4 — Train Linear, Polynomial, and RBF SVMs
# ════════════════════════════════════════════════════════════════════

def train_svm_models(X_train, y_train, C=1.0):
    """
    Train three SVM models with different kernels.

    Parameters
    ----------
    C : Regularization parameter.
        Large C → smaller margin, fewer violations (may overfit).
        Small C → larger margin, more violations allowed (may underfit).

    Returns dict: {'linear': model, 'poly': model, 'rbf': model}
    """
    models = {
        # Linear: K(x,z) = x·z — fastest, needs linear separability
        'linear': SVC(kernel='linear', C=C, random_state=42),

        # Polynomial: K(x,z) = (γx·z + r)^d — degree=3 is common default
        'poly'  : SVC(kernel='poly', C=C, degree=3, coef0=1, random_state=42),

        # RBF: K(x,z) = exp(-γ||x-z||²) — most flexible, gamma='scale' is safe default
        'rbf'   : SVC(kernel='rbf', C=C, gamma='scale', random_state=42),
    }

    print('Training SVM models...')
    for name, model in models.items():
        model.fit(X_train, y_train)
        print(f'  [{name:6s}] ✓ trained | support vectors: {model.support_vectors_.shape[0]}')

    return models


# ── Train on Synthetic Data ────────────────────────────────────────
print('=== Synthetic Dataset ===')
models_syn = train_svm_models(X_tr_syn, y_tr_syn)

# ── Train on Real Data (full features for metrics) ─────────────────
print('\n=== Breast Cancer Dataset (30 features) ===')
models_bc_full = train_svm_models(X_tr_bc, y_tr_bc)

# ── Train on Real Data 2-D slice (for boundary plots only) ─────────
print('\n=== Breast Cancer Dataset (2-D for plots) ===')
models_bc_2d = train_svm_models(X_tr_bc2d, y_tr_bc)

print('\n✅ All models trained.')

---
# Part 3 — Evaluation: Accuracy, Precision, Recall, F1

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 5 — Evaluation Functions
# ════════════════════════════════════════════════════════════════════

def evaluate_model(model, X_test, y_test, name, verbose=True):
    """
    Compute accuracy, precision, recall, F1, and confusion matrix.

    All metrics use weighted averaging to handle class imbalance.
    """
    y_pred = model.predict(X_test)
    metrics = {
        'Kernel'    : name,
        'Accuracy'  : round(accuracy_score(y_test, y_pred), 4),
        'Precision' : round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'Recall'    : round(recall_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'F1-Score'  : round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        '_cm'       : confusion_matrix(y_test, y_pred),
    }
    if verbose:
        print(f'\n  {name}')
        print(classification_report(y_test, y_pred, zero_division=0))
    return metrics


def evaluate_all(models_dict, X_test, y_test, verbose=True):
    """Run evaluate_model on each model, return list of dicts and summary DataFrame."""
    label_map = {'linear': 'Linear', 'poly': 'Polynomial (d=3)', 'rbf': 'RBF'}
    all_results = [evaluate_model(m, X_test, y_test, label_map[k], verbose)
                   for k, m in models_dict.items()]
    df = pd.DataFrame([{k: v for k, v in r.items() if k != '_cm'}
                       for r in all_results]).set_index('Kernel')
    return all_results, df


# ── Evaluate on Synthetic ──────────────────────────────────────────
print('━'*55)
print('EVALUATION — Synthetic Dataset')
print('━'*55)
results_syn, df_syn = evaluate_all(models_syn, X_te_syn, y_te_syn, verbose=True)

print('\n━'*55)
print('EVALUATION — Breast Cancer Dataset (full 30 features)')
print('━'*55)
results_bc, df_bc = evaluate_all(models_bc_full, X_te_bc, y_te_bc, verbose=True)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 6 — Final Summary Tables
# ════════════════════════════════════════════════════════════════════

def print_summary(df, label):
    print(f'\n{"="*60}')
    print(f'  SUMMARY: {label}')
    print(f'{"="*60}')
    print(df.to_string())
    print(f'\n  ★ Best per metric:')
    for col in df.columns:
        best = df[col].idxmax()
        print(f'    {col:<12}: {best} ({df[col].max():.4f})')

print_summary(df_syn, 'Synthetic (make_moons)')
print_summary(df_bc,  'Breast Cancer (Real, 30 features)')

# Display styled tables inline in Colab
print('\n── Synthetic Dataset ──')
display(df_syn.style
        .highlight_max(color='#b7e4c7', axis=0)
        .format(precision=4)
        .set_caption('Synthetic Dataset — Kernel Performance Comparison'))

print('\n── Breast Cancer Dataset ──')
display(df_bc.style
        .highlight_max(color='#b7e4c7', axis=0)
        .format(precision=4)
        .set_caption('Breast Cancer Dataset — Kernel Performance Comparison'))

---
# Part 4 — Visualizations

## 4A. Decision Boundaries — All 3 Kernels

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 7 — Decision Boundary Plotting Helper
# ════════════════════════════════════════════════════════════════════

def plot_decision_boundaries(models_dict, X_train, y_train,
                              title='', fname='boundaries.png', h=0.04):
    """
    Plot filled decision regions + boundary line + data points +
    support vectors for all three kernels side-by-side.

    h : mesh step — smaller = finer grid but slower rendering.
    """
    kernel_labels = {
        'linear': 'Linear Kernel\n(straight hyperplane)',
        'poly'  : 'Polynomial Kernel (d=3)\n(curved boundary)',
        'rbf'   : 'RBF Kernel\n(flexible Gaussian boundary)',
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.01)

    for ax, (key, model) in zip(axes, models_dict.items()):
        # Build mesh grid
        x_min, x_max = X_train[:, 0].min() - 0.6, X_train[:, 0].max() + 0.6
        y_min, y_max = X_train[:, 1].min() - 0.6, X_train[:, 1].max() + 0.6
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                             np.arange(y_min, y_max, h))
        Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        # Shaded decision regions
        ax.contourf(xx, yy, Z, alpha=0.30, cmap=CMAP_LIGHT)
        # Decision boundary line
        ax.contour(xx, yy, Z, colors='black', linewidths=1.0, linestyles='--')

        # Training points coloured by class
        for cls, color in enumerate(CLASS_COLORS):
            mask = y_train == cls
            ax.scatter(X_train[mask, 0], X_train[mask, 1],
                       c=color, edgecolors='white', linewidths=0.4,
                       s=35, label=f'Class {cls}', alpha=0.85, zorder=3)

        # Support vectors — highlighted with open circle
        sv = model.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1],
                   s=120, facecolors='none', edgecolors='black',
                   linewidths=1.5, zorder=5, label=f'Support Vectors ({sv.shape[0]})')

        ax.set_title(kernel_labels.get(key, key), fontsize=12, fontweight='bold', pad=8)
        ax.set_xlabel('Feature 1 (scaled)', fontsize=9)
        ax.set_ylabel('Feature 2 (scaled)', fontsize=9)
        ax.legend(fontsize=7.5, loc='lower right')

    plt.tight_layout()
    path = f'{OUTPUT_DIR}/{fname}'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {path}')


# ── Plot for Synthetic Dataset ─────────────────────────────────────
plot_decision_boundaries(
    models_syn, X_tr_syn, y_tr_syn,
    title='SVM Decision Boundaries — Synthetic (make_moons)',
    fname='decision_boundaries_synthetic.png'
)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 8 — Decision Boundaries: Breast Cancer (2-D slice)
# ════════════════════════════════════════════════════════════════════

plot_decision_boundaries(
    models_bc_2d, X_tr_bc2d, y_tr_bc,
    title='SVM Decision Boundaries — Breast Cancer (first 2 features)',
    fname='decision_boundaries_breast_cancer.png',
    h=0.05
)

---
## 4B. Effect of Gamma (γ) on RBF Decision Boundary

### What does the Kernel Trick achieve computationally?

> Instead of explicitly computing the feature mapping $\phi(x)$ (which for RBF is *infinite*-dimensional), the kernel evaluates $K(x, z) = \langle \phi(x), \phi(z) \rangle$ directly in $O(n)$ time using the Gaussian formula. This makes computationally impossible high-dimensional transformations feasible — the model never needs to store or compute $\phi(x)$ explicitly.

**Gamma (γ)** controls the *reach* of a single training point's influence:
- **Small γ** → wide Gaussian → smooth, global boundary → risk of **underfitting**
- **Large γ** → narrow Gaussian → boundary wraps tightly around training points → risk of **overfitting**

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 9 — Gamma Effect Visualisation
# ════════════════════════════════════════════════════════════════════

gamma_configs = {
    'small' : {'g': 0.01,  'label': 'Small γ = 0.01\n(Smooth → Underfitting)'},
    'medium': {'g': 1.0,   'label': 'Medium γ = 1.0\n(Balanced → Good fit)'},
    'large' : {'g': 100.0, 'label': 'Large γ = 100\n(Complex → Overfitting)'},
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Effect of Gamma (γ) on RBF Kernel Decision Boundary (C=10)',
             fontsize=15, fontweight='bold', y=1.01)

for ax, (key, cfg) in zip(axes, gamma_configs.items()):
    # Train a fresh RBF model with this gamma
    model = SVC(kernel='rbf', C=10, gamma=cfg['g'], random_state=42)
    model.fit(X_tr_syn, y_tr_syn)

    # Mesh
    x_min, x_max = X_tr_syn[:, 0].min() - 0.6, X_tr_syn[:, 0].max() + 0.6
    y_min, y_max = X_tr_syn[:, 1].min() - 0.6, X_tr_syn[:, 1].max() + 0.6
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.04),
                         np.arange(y_min, y_max, 0.04))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.30, cmap=CMAP_LIGHT)
    ax.contour(xx, yy, Z, colors='black', linewidths=1.0, linestyles='--')

    for cls, color in enumerate(CLASS_COLORS):
        mask = y_tr_syn == cls
        ax.scatter(X_tr_syn[mask, 0], X_tr_syn[mask, 1],
                   c=color, edgecolors='white', linewidths=0.4,
                   s=35, alpha=0.85, zorder=3)

    sv = model.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1],
               s=120, facecolors='none', edgecolors='black',
               linewidths=1.5, zorder=5)

    train_acc = model.score(X_tr_syn, y_tr_syn)
    test_acc  = model.score(X_te_syn, y_te_syn)
    ax.set_title(cfg['label'], fontsize=12, fontweight='bold', pad=8)
    ax.set_xlabel('Feature 1 (scaled)', fontsize=9)
    ax.set_ylabel('Feature 2 (scaled)', fontsize=9)

    # Show train vs test accuracy and support vector count in each subplot
    info = f'SVs: {sv.shape[0]}\nTrain: {train_acc:.3f}\nTest : {test_acc:.3f}'
    ax.text(0.03, 0.97, info, transform=ax.transAxes, fontsize=9,
            va='top', bbox=dict(boxstyle='round,pad=0.3',
                                facecolor='lightyellow', alpha=0.85))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/gamma_effect_rbf.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {OUTPUT_DIR}/gamma_effect_rbf.png')

---
## 4C. GridSearchCV — Tuning C and Gamma for RBF

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 10 — GridSearchCV: Tuning RBF Hyperparameters
# ════════════════════════════════════════════════════════════════════

param_grid = {
    'C'    : [0.01, 0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 10]
}

# 5-fold cross-validation; n_jobs=-1 uses all CPU cores
grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid=param_grid, cv=5,
    scoring='accuracy', n_jobs=-1, verbose=0
)
grid_search.fit(X_tr_syn, y_tr_syn)

best_params = grid_search.best_params_
best_cv     = grid_search.best_score_
best_rbf    = grid_search.best_estimator_

print(f'Best hyperparameters : C={best_params["C"]}, gamma={best_params["gamma"]}')
print(f'Best CV accuracy     : {best_cv:.4f}')
print(f'Test accuracy (tuned): {best_rbf.score(X_te_syn, y_te_syn):.4f}')

# Extract results and pivot into C × gamma table
cv_df = pd.DataFrame(grid_search.cv_results_)[['param_C', 'param_gamma', 'mean_test_score']]
pivot = cv_df.pivot_table(index='param_C', columns='param_gamma',
                           values='mean_test_score')

# ── GridSearch Heatmap ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean CV Accuracy'})
ax.set_title('GridSearchCV — RBF Kernel: C vs Gamma (Mean 5-Fold CV Accuracy)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Gamma (γ)', fontsize=10)
ax.set_ylabel('C (Regularization)', fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/gridsearch_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {OUTPUT_DIR}/gridsearch_heatmap.png')

# Show top 5 combinations
print('\nTop 5 hyperparameter combinations:')
display(cv_df.sort_values('mean_test_score', ascending=False)
        .head(5).reset_index(drop=True))

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 11 — Metric Comparison Bar Charts (both datasets)
# ════════════════════════════════════════════════════════════════════

def plot_metric_bar(df, title, fname):
    """
    Grouped bar chart comparing Accuracy and F1-Score
    across all three kernels. Value labels placed on each bar.
    """
    metrics = ['Accuracy', 'F1-Score']
    kernels = df.index.tolist()
    x       = np.arange(len(kernels))
    width   = 0.30
    colors  = ['#5C85D6', '#E07B54']   # blue=Accuracy, orange=F1

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        vals   = df[metric].values
        offset = (i - 0.5) * width
        bars   = ax.bar(x + offset, vals, width, label=metric,
                        color=color, edgecolor='white', linewidth=0.8)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(kernels, fontsize=11)
    ax.set_ylim(0, 1.18)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.legend(fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    path = f'{OUTPUT_DIR}/{fname}'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {path}')


plot_metric_bar(df_syn, 'Kernel Performance — Synthetic (make_moons)',
                'metric_comparison_synthetic.png')

plot_metric_bar(df_bc, 'Kernel Performance — Breast Cancer (Real)',
                'metric_comparison_breast_cancer.png')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 12 — Confusion Matrices (both datasets)
# ════════════════════════════════════════════════════════════════════

def plot_confusion_matrices(results, title_suffix, fname):
    """Side-by-side seaborn heatmap confusion matrices for all 3 kernels."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Confusion Matrices — {title_suffix}',
                 fontsize=14, fontweight='bold', y=1.02)

    for ax, res in zip(axes, results):
        cm = res['_cm']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    linewidths=0.5, linecolor='gray',
                    ax=ax, cbar=False)
        ax.set_title(res['Kernel'], fontsize=12, pad=8)
        ax.set_xlabel('Predicted Label', fontsize=9)
        ax.set_ylabel('True Label', fontsize=9)

    plt.tight_layout()
    path = f'{OUTPUT_DIR}/{fname}'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {path}')


plot_confusion_matrices(results_syn, 'Synthetic (make_moons)',
                        'confusion_matrices_synthetic.png')

plot_confusion_matrices(results_bc, 'Breast Cancer (Real)',
                        'confusion_matrices_breast_cancer.png')

---
# Part 5 — Final Summary & Conclusions

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 13 — Final Summary
# ════════════════════════════════════════════════════════════════════

print('█'*60)
print('  FINAL PROJECT SUMMARY')
print('█'*60)

print_summary(df_syn, 'Synthetic Dataset (make_moons)')
print_summary(df_bc,  'Real Dataset (Breast Cancer, 30 features)')

best_syn  = df_syn['F1-Score'].idxmax()
best_real = df_bc['F1-Score'].idxmax()

print(f'\n{"─"*60}')
print(f'  🏆 Best kernel — Synthetic : {best_syn}')
print(f'  🏆 Best kernel — Real      : {best_real}')
print(f'{"─"*60}')

print(f'''
  📌 CONCLUSION:

  • RBF kernel wins on both datasets (highest F1-Score) because it
    uses the Gaussian function to implicitly map data into an
    infinite-dimensional space, handling arbitrary non-linear boundaries.

  • Linear kernel fails on make_moons because no straight line can
    separate the two interleaving half-circles. On linearly-structured
    real data (Breast Cancer), it still performs competitively.

  • Polynomial (d=3) is a solid middle ground — it captures curved
    boundaries but can overfit or underfit if degree is wrong.

  • Gamma matters critically for RBF:
    - γ too small → boundary too smooth → underfitting
    - γ too large → boundary too wiggly → overfitting
    - GridSearchCV found best: C={best_params['C']}, γ={best_params['gamma']}
''')

# List all saved files
print(f'\n📁 All plots saved to ./{OUTPUT_DIR}/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) // 1024
    print(f'   📊 {f:<50} ({size_kb} KB)')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 14 — Download All Output PNGs as a ZIP (Colab only)
# ════════════════════════════════════════════════════════════════════

import shutil
from google.colab import files   # This cell only works in Google Colab

# Create a ZIP archive of all outputs
shutil.make_archive('svm_outputs', 'zip', OUTPUT_DIR)
files.download('svm_outputs.zip')
print('✅ Download started: svm_outputs.zip')